Load Data into SQL

In [1]:
import pandas as pd
import sqlite3

Load Cleaned Data

In [2]:
users = pd.read_csv("../data/processed/users_cleaned.csv")
transactions = pd.read_csv("../data/processed/transactions_cleaned.csv")
campaigns = pd.read_csv("../data/processed/campaigns_cleaned.csv")

Create Database

In [3]:
conn = sqlite3.connect("../data/fintech.db")

Load Tables into SQL

In [4]:
users.to_sql("users", conn, if_exists="replace", index=False)
transactions.to_sql("transactions", conn, if_exists="replace", index=False)
campaigns.to_sql("campaigns", conn, if_exists="replace", index=False)

10

In [5]:
pd.read_sql("SELECT * FROM users LIMIT 5", conn)

,user_id,signup_date,country,age,gender,acquisition_channel,campaign_id,kyc_completed,kyc_date,first_transaction_date,total_transactions,total_amount,avg_transaction_value,is_converted,is_active_user,kyc_done_flag,transaction_done_flag,cohort_month
0,USR_1,2023-04-13,India,25,Male,Referral,CMP_4,1,2023-04-16,2023-05-23,10.0,44526.59,4452.659000,1,1,1,1,2023-04
1,USR_2,2023-08-03,UK,57,Female,Google,CMP_8,1,2023-08-05,2023-09-13,5.0,29532.43,5906.486000,1,1,1,1,2023-08
2,USR_3,2023-12-10,Canada,38,Male,Facebook,CMP_6,1,2023-12-11,2023-12-23,48.0,226313.42,4714.862917,1,1,1,1,2023-12
3,USR_4,2023-08-24,Canada,59,Female,Google,CMP_9,0,NaN,NaN,0.0,0.00,0.000000,0,0,0,0,2023-08
4,USR_5,2023-09-28,UK,20,Male,Google,CMP_3,1,2023-09-29,NaN,0.0,0.00,0.000000,0,0,1,0,2023-09


ANALYSIS 1 — Funnel Conversion

Funnel Query

In [6]:
query = """
SELECT
    COUNT(*) AS total_users,
    
    SUM(kyc_done_flag) AS kyc_completed_users,
    
    SUM(transaction_done_flag) AS converted_users,
    
    ROUND(100.0 * SUM(kyc_done_flag) / COUNT(*), 2) AS kyc_conversion_rate,
    
    ROUND(100.0 * SUM(transaction_done_flag) / COUNT(*), 2) AS overall_conversion_rate

FROM users
"""

funnel_df = pd.read_sql(query, conn)
funnel_df

,total_users,kyc_completed_users,converted_users,kyc_conversion_rate,overall_conversion_rate
0,5000,3983,3681,79.66,73.62


ANALYSIS 2 — Campaign Performance

In [7]:
query = """
SELECT
    c.campaign_name,
    COUNT(u.user_id) AS total_users,
    SUM(u.transaction_done_flag) AS converted_users,
    
    ROUND(100.0 * SUM(u.transaction_done_flag) / COUNT(u.user_id), 2) AS conversion_rate

FROM users u
JOIN campaigns c
ON u.campaign_id = c.campaign_id

GROUP BY c.campaign_name
ORDER BY conversion_rate DESC
"""

campaign_perf = pd.read_sql(query, conn)
campaign_perf

,campaign_name,total_users,converted_users,conversion_rate
0,Campaign_1,531,403,75.89
1,Campaign_7,498,377,75.70
2,Campaign_8,468,350,74.79
3,Campaign_9,474,354,74.68
4,Campaign_4,484,360,74.38
5,Campaign_5,490,360,73.47
6,Campaign_3,548,400,72.99
7,Campaign_10,538,389,72.30
8,Campaign_2,475,340,71.58
9,Campaign_6,494,348,70.45


ANALYSIS 3 — User Activity Segmentation

In [8]:
query = """
SELECT
    CASE
        WHEN total_transactions = 0 THEN 'Inactive'
        WHEN total_transactions BETWEEN 1 AND 5 THEN 'Low Activity'
        WHEN total_transactions BETWEEN 6 AND 20 THEN 'Medium Activity'
        ELSE 'High Activity'
    END AS user_segment,
    
    COUNT(*) AS user_count

FROM users

GROUP BY user_segment
ORDER BY user_count DESC
"""

segment_df = pd.read_sql(query, conn)
segment_df

,user_segment,user_count
0,Medium Activity,2042
1,Inactive,1319
2,Low Activity,1249
3,High Activity,390


Retention Analysis (Cohort Analysis)

Get required data

In [55]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("../data/fintech.db")

query = """
SELECT 
    u.user_id,
    u.signup_date,
    t.transaction_date
FROM users u
JOIN transactions t
ON u.user_id = t.user_id
WHERE t.status = 'success'
"""

df = pd.read_sql(query, conn)

In [56]:
df["signup_date"] = pd.to_datetime(df["signup_date"])
df["transaction_date"] = pd.to_datetime(df["transaction_date"])

Create Cohort Columns

In [57]:
df["cohort_month"] = df["signup_date"].dt.to_period("M")
df["transaction_month"] = df["transaction_date"].dt.to_period("M")

Cohort Index


In [58]:
df["cohort_index"] = (
    (df["transaction_month"].dt.year - df["cohort_month"].dt.year) * 12 +
    (df["transaction_month"].dt.month - df["cohort_month"].dt.month)
)

In [59]:
#Remove invalid rows
df = df[df["cohort_index"] >= 0]

In [60]:
#Remove duplicate users
df_unique = df.drop_duplicates(subset=["user_id", "cohort_month", "cohort_index"])

In [61]:
df_unique.duplicated(subset=["user_id", "cohort_month", "cohort_index"]).sum()

np.int64(0)

In [62]:
#Cohert table
cohort_data = df_unique.groupby(
    ["cohort_month", "cohort_index"]
)["user_id"].nunique().reset_index()

In [63]:
#Pivot
cohort_pivot = cohort_data.pivot_table(
    index="cohort_month",
    columns="cohort_index",
    values="user_id"
)

Retention %

In [64]:
users = pd.read_sql("SELECT user_id, signup_date FROM users", conn)

users["signup_date"] = pd.to_datetime(users["signup_date"])

cohort_size = users.groupby(
    users["signup_date"].dt.to_period("M")
)["user_id"].nunique()

cohort_size.index.name = "cohort_month"

In [65]:
retention = cohort_pivot.divide(cohort_size, axis=0)

retention = retention.round(3)
retention

cohort_index,0,1,2,3,4,5,6
cohort_month,,,,,,,
2023-01,0.386,0.566,0.576,0.568,0.568,0.547,0.374
2023-02,0.319,0.610,0.560,0.558,0.558,0.545,0.327
2023-03,0.349,0.549,0.565,0.574,0.535,0.586,0.307
2023-04,0.355,0.587,0.550,0.572,0.570,0.584,0.311
2023-05,0.371,0.514,0.545,0.568,0.545,0.538,0.322
2023-06,0.371,0.604,0.576,0.573,0.576,0.597,0.298
2023-07,0.351,0.577,0.552,0.566,0.568,0.575,0.310
2023-08,0.439,0.584,0.600,0.602,0.597,0.613,0.299
2023-09,0.364,0.538,0.567,0.564,0.521,0.531,0.300


In [67]:
retention[0] = 1.0

In [68]:
retention = retention.sort_index(axis=1)

In [69]:
retention = retention.round(3)
retention

cohort_index,0,1,2,3,4,5,6
cohort_month,,,,,,,
2023-01,1.0,0.566,0.576,0.568,0.568,0.547,0.374
2023-02,1.0,0.610,0.560,0.558,0.558,0.545,0.327
2023-03,1.0,0.549,0.565,0.574,0.535,0.586,0.307
2023-04,1.0,0.587,0.550,0.572,0.570,0.584,0.311
2023-05,1.0,0.514,0.545,0.568,0.545,0.538,0.322
2023-06,1.0,0.604,0.576,0.573,0.576,0.597,0.298
2023-07,1.0,0.577,0.552,0.566,0.568,0.575,0.310
2023-08,1.0,0.584,0.600,0.602,0.597,0.613,0.299
2023-09,1.0,0.538,0.567,0.564,0.521,0.531,0.300


In [70]:
retention.max().max()

np.float64(1.0)

In [71]:
retention.to_csv("../data/processed/retention.csv")